# muon_db Catalog And Layer Guide

This notebook is the readable entry point for the local `muon_db` lakehouse. It registers the Spark-written Parquet folders as DuckDB views, lists the available tables, and documents what each layer contains.

The ETL flow implemented in this workspace is:

```text
ROOT NanoAOD files
  -> converted Parquet files
  -> bronze raw projections
  -> silver cleaned physics objects
  -> gold curated analytical datasets
```

AI, ML tensors, GNN graphs, anomaly models, and feature-store serving are intentionally outside this implemented data-engineering scope.

In [ ]:
from pathlib import Path
import sys

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent
sys.path.insert(0, str(WORKSPACE))

from src.access.muon_db import connect_with_tables

MUON_DB_ROOT = WORKSPACE / "data" / "muon_db"
connection, tables = connect_with_tables(MUON_DB_ROOT)
MUON_DB_ROOT

## Table Registry

The access layer creates DuckDB views over these Parquet folders. The files are still physically stored under `data/muon_db`; DuckDB is only used for local querying and notebook exploration.

In [ ]:
import pandas as pd

catalog_rows = []
for table in tables:
    row_count = connection.execute(f"SELECT count(*) FROM {table.name}").fetchone()[0]
    catalog_rows.append({
        "layer": table.layer,
        "table": table.name,
        "rows": row_count,
        "path": str(table.path.relative_to(WORKSPACE)),
    })

catalog = pd.DataFrame(catalog_rows)
catalog

## Bronze Layer

Purpose: preserve selected NanoAOD branches with minimal transformation after ROOT-to-Parquet conversion.

Business logic applied:

- Project only the curated branch set needed for the first lakehouse build.
- Add dataset metadata: dataset name, CMS dataset path, year, run period, NanoAOD version.
- Add lineage fields: source Parquet file and ingestion timestamp.
- Add deterministic `event_id` from dataset, year, run, luminosity block, and event number.

Tables:

- `bronze_event`: event identifiers and lineage.
- `bronze_muon`: raw muon collection arrays.
- `bronze_jet`: raw jet collection arrays.
- `bronze_met`: raw event-level MET fields.
- `bronze_trigger`: trigger decisions and event quality flags.

Bronze object tables intentionally keep event-shaped nested/list columns. Object flattening happens in silver.

In [ ]:
connection.execute("DESCRIBE bronze_event").fetchdf()

In [ ]:
connection.execute("SELECT * FROM bronze_event LIMIT 5").fetchdf()

## Silver Layer

Purpose: transform raw NanoAOD-shaped data into cleaned, normalized physics tables.

Business logic applied:

- Keep events that pass detector quality: `Flag_goodVertices == true` and `Flag_METFilters == true`.
- Keep events that pass the SingleMuon trigger requirement: `HLT_IsoMu24 == true`.
- Flatten muons into one row per muon.
- Keep cleaned muons with `tight_id == true`, `isolation < 0.15`, and `pt > 20`.
- Flatten jets into one row per jet and keep positive-pT jets.
- Standardize object field names from NanoAOD branch names to analysis-friendly names.

Tables:

- `silver_event`: cleaned event-level rows.
- `silver_muon`: one row per cleaned muon.
- `silver_jet`: one row per jet.
- `silver_met`: cleaned event-level MET.
- `silver_trigger`: trigger and quality decision record per cleaned event.

In [ ]:
connection.execute("DESCRIBE silver_muon").fetchdf()

In [ ]:
connection.execute("""
SELECT event_id, muon_idx, pt, eta, phi, mass, charge, isolation, tight_id
FROM silver_muon
ORDER BY pt DESC
LIMIT 10
""").fetchdf()

## Gold Layer

Purpose: create curated analytical datasets for physics analysis and dashboard-style exploration.

Business logic applied:

- `event_summary`: one row per cleaned event, with object counts, leading pT values, `MET_pt`, `HT`, and `ST`.
- `dimuon`: opposite-sign muon pairs from cleaned muons, with invariant mass and angular separation.
- `jet`: curated jet kinematics and b-tag score.
- `met`: curated MET fields.

Definitions:

- `HT`: scalar sum of jet pT per event.
- `ST`: scalar sum of jet pT, cleaned muon pT, and MET per event.
- `delta_r`: angular distance between muons, `sqrt(delta_eta^2 + delta_phi^2)`.
- `invariant_mass`: four-vector dimuon mass using muon pT, eta, phi, and mass.

In [ ]:
connection.execute("DESCRIBE event_summary").fetchdf()

In [ ]:
connection.execute("""
SELECT event_id, n_muons, n_jets, leading_muon_pt, leading_jet_pt, MET_pt, HT, ST
FROM event_summary
ORDER BY ST DESC
LIMIT 10
""").fetchdf()

## Layer Reduction Summary

This shows how event and object counts change as filters and transformations are applied.

In [ ]:
connection.execute("""
SELECT 'bronze_event' AS table_name, count(*) AS rows FROM bronze_event
UNION ALL SELECT 'silver_event', count(*) FROM silver_event
UNION ALL SELECT 'silver_muon', count(*) FROM silver_muon
UNION ALL SELECT 'silver_jet', count(*) FROM silver_jet
UNION ALL SELECT 'event_summary', count(*) FROM event_summary
UNION ALL SELECT 'dimuon', count(*) FROM dimuon
""").fetchdf()